In [2]:
import os
import ast
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# Load API keys from root .env
load_dotenv(dotenv_path="../../../.env")
if not os.getenv("OPENROUTER_API_KEY"):
    load_dotenv()

# Store all available keys in a list
api_keys = []
key1 = os.getenv("OPENROUTER_API_KEY")
key2 = os.getenv("OPENROUTER_API_KEY_NEW")

if key1: api_keys.append(key1)
if key2: api_keys.append(key2)

if not api_keys:
    raise ValueError("No API keys found! Please verify your .env file.")

# Global variable to track which key we are currently using
active_key_index = 0

# Set model to GPT-5.6 Luna
MODEL = "openai/gpt-5.6-luna"

print(f"Environment ready. Loaded {len(api_keys)} API keys.")
print(f"Target Model set to: {MODEL}")

Environment ready. Loaded 2 API keys.
Target Model set to: openai/gpt-5.6-luna


In [3]:
INPUT_FILE = "../../../dataset/NER/test_tag_columns.csv"  
OUTPUT_FILE = "gpt5.6_luna_specialist_full_dataset.csv"
BACKUP_FILE = "backup_" + OUTPUT_FILE

def extract_first_entity(val):
    """Safely extracts the first valid entity from various column formats."""
    if pd.isna(val): return None
    if isinstance(val, list):
        return str(val[0]).strip() if len(val) > 0 and str(val[0]).strip() else None

    val_str = str(val).strip()
    if not val_str or val_str in ["[]", "nan", "None", ""]: return None

    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            parsed = ast.literal_eval(val_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                first_elem = str(parsed[0]).strip()
                return first_elem if first_elem else None
        except Exception:
            pass
    return val_str

print(f"Loading input file: {INPUT_FILE}...")
df_raw = pd.read_csv(INPUT_FILE)

# Extract and clean the Specialist entity
df_raw["target_specialist"] = df_raw["Specialist"].apply(extract_first_entity)

# Filter for rows that actually have a Specialist
df_valid = df_raw.dropna(subset=["target_specialist"]).copy()
df_valid = df_valid[df_valid["target_specialist"] != ""].reset_index(drop=True)

df_full = df_valid.copy()

print(f"Total valid Specialist rows found: {len(df_full)}")
print(f"Ready to process ALL {len(df_full)} rows.")

Loading input file: ../../../dataset/NER/test_tag_columns.csv...
Total valid Specialist rows found: 650
Ready to process ALL 650 rows.


In [4]:
PROMPT_TEMPLATE = """You are given a Bangla medical sentence containing one or more medical specialist entities.

Your task is to create a modified version of the sentence by replacing exactly ONE medical specialist entity with a different but closely related medical specialist.

Rules:
1. Identify the specified medical specialist entity in the sentence.
2. Replace exactly ONE occurrence of that medical specialist with another medical specialist.
3. The replacement must be different from the original medical specialist.
4. The replacement must NOT be another medical specialist already present in the original sentence.
5. The replacement should be medically plausible and closely related to the original specialist in terms of medical field, treated body system, or clinical context (e.g., closely related departments like Cardiology and Pulmonology).
6. Prefer a specialist that could reasonably be consulted for similar, overlapping, or related clinical issues.
7. Do NOT replace the medical specialist with an unrelated one simply because it is common.
8. The replacement must fit naturally into the surrounding sentence without making the sentence medically or linguistically implausible.
9. Do not add any additional medical specialist.
10. Do not remove, add, or modify any other information in the sentence.
11. Keep the rest of the sentence exactly as unchanged as possible.
12. Do not modify the original NER annotation.
13. Do not provide explanations or identify the replacement in prose.
14. Return your answer in EXACTLY the following format:

Modified Specialist: <new specialist>

Modified Sentence: <modified Bangla sentence>

Example:

Original sentence:
আমার বাবার জন্য একজন ভালো হৃদরোগ বিশেষজ্ঞ খুঁজছি।

Specialist entity:
হৃদরোগ বিশেষজ্ঞ

A suitable replacement should be a medically related medical specialist, rather than an arbitrary or unrelated specialist.

Output:

Modified Specialist: বক্ষব্যাধি বিশেষজ্ঞ

Modified Sentence: আমার বাবার জন্য একজন ভালো বক্ষব্যাধি বিশেষজ্ঞ খুঁজছি।

Now perform the replacement.

Original sentence:
{sentence}

Specialist entity:
{specialist}
"""

def replace_specialist(sentence, specialist):
    global active_key_index
    
    prompt = PROMPT_TEMPLATE.format(
        sentence=str(sentence).strip(), 
        specialist=str(specialist).strip()
    )

    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    # Dual-Key Rotation Loop
    for _ in range(len(api_keys)):
        current_key = api_keys[active_key_index]
        
        headers = {
            "Authorization": f"Bearer {current_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        
        # 402 = Payment Required / 429 = Rate Limited
        if response.status_code in [402, 429]:
            print(f"   [!] Key {active_key_index + 1} hit a limit (Status {response.status_code}). Swapping keys...")
            active_key_index = (active_key_index + 1) % len(api_keys)
            continue
            
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"].strip()

        modified_specialist = ""
        modified_sentence = ""

        # Parse the exact format requested in Rule 14
        for line in content.splitlines():
            if line.startswith("Modified Specialist:"):
                modified_specialist = line.replace("Modified Specialist:", "").strip()
            elif line.startswith("Modified Sentence:"):
                modified_sentence = line.replace("Modified Sentence:", "").strip()

        return modified_specialist, modified_sentence
        
    raise Exception("CRITICAL ERROR: ALL API keys have reached their limits!")

In [5]:
# 1. LOAD BACKUP IF IT EXISTS
if os.path.exists(BACKUP_FILE):
    print(f"Found backup file: {BACKUP_FILE}")
    df_backup = pd.read_csv(BACKUP_FILE)
    start_index = len(df_backup)
    
    modified_specialists = df_backup["modified_specialist"].tolist()
    modified_sentences = df_backup["modified_sentence"].tolist()
    print(f"Resuming generation from row {start_index + 1}...\n")
else:
    print("No backup file found. Starting from row 1...\n")
    start_index = 0
    modified_specialists = []
    modified_sentences = []

print(f"Starting execution loop using {MODEL}...\n")

# 2. RUN THE LOOP
for i in range(start_index, len(df_full)):
    row = df_full.iloc[i]
    sentence = row["text"]
    specialist = row["target_specialist"]

    print(f"[{i+1}/{len(df_full)}] Swapping: '{specialist}'...")

    try:
        new_specialist, new_sentence = replace_specialist(sentence, specialist)
        print(f"   -> Result: '{new_specialist}'")
    except Exception as e:
        print(f"   -> Error: {e}")
        new_specialist = ""
        new_sentence = ""

    modified_specialists.append(new_specialist)
    modified_sentences.append(new_sentence)

    time.sleep(0.5)

    # 3. AUTO-SAVE LOGIC
    if (i + 1) % 50 == 0:
        df_temp = df_full.loc[:i].copy()
        df_temp["modified_specialist"] = modified_specialists
        df_temp["modified_sentence"] = modified_sentences
        df_temp.to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")
        print(f"   --- Auto-saved backup at row {i+1} ---")

# 4. FINAL SAVE
df_full["modified_specialist"] = modified_specialists
df_full["modified_sentence"] = modified_sentences

export_cols = [
    "text",
    "Specialist",
    "target_specialist",
    "modified_specialist",
    "modified_sentence",
]
available_export_cols = [c for c in export_cols if c in df_full.columns]

df_full[available_export_cols].to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\nExecution complete! Final dataset saved to: {OUTPUT_FILE}")

No backup file found. Starting from row 1...

Starting execution loop using openai/gpt-5.6-luna...

[1/650] Swapping: 'শিশু বিশেষজ্ঞ'...
   -> Result: 'শিশু শ্বাসতন্ত্র বিশেষজ্ঞ'
[2/650] Swapping: 'Heart er doctors'...
   -> Result: 'Pulmonologist'
[3/650] Swapping: 'নিউরোলজিস্ট | নিউরোমেডিসিন বিশেষজ্ঞ'...
   -> Result: 'মুভমেন্ট ডিসঅর্ডার বিশেষজ্ঞ'
[4/650] Swapping: 'নিউরোমেডিসিন বিশেষজ্ঞ'...
   -> Result: 'নিউরোলজি বিশেষজ্ঞ'
[5/650] Swapping: 'মেডিসিন বিশেষজ্ঞ'...
   -> Result: 'নাক-কান-গলা বিশেষজ্ঞ'
[6/650] Swapping: 'Neuro - Medicine বিশেষজ্ঞ'...
   -> Result: 'নিউরোলজি বিশেষজ্ঞ'
[7/650] Swapping: 'gynaecologist'...
   -> Result: 'obstetrician'
[8/650] Swapping: 'পুস্তিবিদ'...
   -> Result: 'ডায়েটিশিয়ান'
[9/650] Swapping: 'paediatrician'...
   -> Result: 'general physician'
[10/650] Swapping: 'চর্মরোগ বিশেষজ্ঞ'...
   -> Result: 'এলার্জি বিশেষজ্ঞ'
[11/650] Swapping: 'skin specialist doctor'...
   -> Result: 'dermatologist'
[12/650] Swapping: 'চর্ম রোগের ডাক্তারকে'...
   -> Result